Предобработка данных о поездках (rides)


In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

df = pd.read_csv('rides.csv')
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

print("Исходный размер датасета:", df.shape)
display(df.head(5))

Предобработка данных о поездках (rides)

In [ ]:
len_before = len(df)
dupl_id = df['id'].duplicated().sum()
if dupl_id > 0:
    df = df.drop_duplicates(subset=['id'], keep='first')

Приведение номинальных столбцов к единому регистру

In [ ]:
for col in ['start_district', 'end_district', 'start_location', 'end_location']:
    df[col] = df[col].astype(str).str.strip().str.title()

display(df['start_district'].unique())

Преобразование дат и расчет длительности

In [ ]:
df['start_date'] = pd.to_datetime(df['start_date'])
df['end_date'] = pd.to_datetime(df['end_date'])
df['duration'] = (df['end_date'] - df['start_date']).dt.total_seconds() / 60

Анализ пропусков и выбросов (с контролем потерь)

In [ ]:
# Пропуски в end_date
len_before = len(df)
df = df.dropna(subset=['end_date'])

# Пропуски в distance
len_before = len(df)
df = df.dropna(subset=['distance'])

# Длительность > 0
len_before = len(df)
df = df[df['duration'] > 0]

# Фильтр по длительности: от 0.5 до 240 минут
len_before = len(df)
df = df[(df['duration'] >= 0.5) & (df['duration'] <= 240)]

# Фильтр по дистанции: от 50 м до 20 км
len_before = len(df)
df = df[df['distance'] <= 20000]
df = df[df['distance'] >= 50]


Расчет стоимости поездки (с учетом другой цены в выходные)

In [ ]:
def get_price_per_minute(row):
    weekday = row['start_date'].weekday()
    hour = row['start_date'].hour
    is_weekend = weekday >= 5

    if 1 <= hour < 6:
        price = 3
    elif 6 <= hour < 10:
        price = 4
    elif 10 <= hour < 16:
        price = 5
    elif 16 <= hour < 22:
        price = 6
    else:
        price = 7

    if is_weekend:
        price += 1
    return price

df['price'] = df.apply(get_price_per_minute, axis=1)

df['start_fee'] = 30.0

mask_promo = (df['start_date'].dt.weekday == 0) & (df['start_date'].dt.hour.between(6,9)) & (df['promo'] == 1)
df.loc[mask_promo, 'start_fee'] = 0.0

df['total_cost'] = df['start_fee'] + df['duration'] * df['price']

display(df[['id', 'start_date', 'duration', 'price', 'start_fee', 'total_cost']].head(10))

Сохранение очищенного датасета

In [ ]:
df.to_csv('rides_clean.csv', index=False)